## 1. Find the inputs and choose a fresh output folder

We start from the original CSV and compare our duplicate removal with Notebook 01's saved exact-deduplicated file. Change INPUT_RUN_TAG if you used another name.

The automatic run name prevents older outputs from being overwritten. A manifest starts with status **in_progress** and changes only after all checks and exports finish. If a cell fails, fix the cause and restart with a fresh run.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dataset/events.csv").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Open this notebook inside the Recommendation project."
INPUT_RUN_TAG = "notebook_run_2026_10_13"
RAW_FILE = PROJECT_ROOT / "dataset/events.csv"
PREVIOUS_FILE = PROJECT_ROOT / "data/interim" / INPUT_RUN_TAG / "events_exact_deduplicated.csv"
NOTEBOOK_FILE = PROJECT_ROOT / "notebooks/07_build_basket_dataset.ipynb"
CONFIG_FILE = PROJECT_ROOT / "configs/datasets.json"
for path in [RAW_FILE, PREVIOUS_FILE, NOTEBOOK_FILE, CONFIG_FILE]:
    assert path.is_file(), f"Missing input: {path}"

RUN_TAG = "basket_prepared_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
DATA_DIR = PROJECT_ROOT / "data/interim" / RUN_TAG
REPORT_DIR = PROJECT_ROOT / "reports" / RUN_TAG
assert not DATA_DIR.exists() and not REPORT_DIR.exists()
DATA_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR.mkdir(parents=True, exist_ok=False)
MANIFEST_FILE = REPORT_DIR / "manifest.json"

FIELDS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
KEYS = ["visitorid", "timestamp"]
EVENT_TYPES = ["view", "addtocart", "transaction"]
DTYPES = {"timestamp": "int64", "visitorid": "int64", "itemid": "int64",
          "event": "string", "transactionid": "Int64"}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_checksums = {str(p.relative_to(PROJECT_ROOT)): file_sha256(p)
                   for p in [RAW_FILE, PREVIOUS_FILE, NOTEBOOK_FILE, CONFIG_FILE]}
manifest = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "in_progress",
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "input_sha256": input_checksums,
    "settings": {
        "eligible_events": EVENT_TYPES, "exact_duplicates": "keep_first_raw_source_row",
        "block_key": KEYS, "block_id_format": "v{visitorid}_t{timestamp}",
        "within_block_order": "unordered",
        "item_membership": "distinct itemid per block",
        "minimum_interaction_filter_applied": False,
        "split_applied": False, "model_representation": "not yet decided",
        "seed": None, "hardware_used": "CPU; no GPU training",
    },
    "data_directory": str(DATA_DIR), "report_directory": str(REPORT_DIR),
    "outputs": [],
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Prepared dataset:", DATA_DIR)
print("Audits:", REPORT_DIR)

Prepared dataset: E:\ALL MCP Servers\Recommendation\data\interim\basket_prepared_20261014_134756_146630
Audits: E:\ALL MCP Servers\Recommendation\reports\basket_prepared_20261014_134756_146630


## 2. Remove exact copies and preserve source links

**drop_duplicates** compares all five original fields and retains the first copy. The source-row number is the original CSV line number, with the header at line 1. It helps you find an event again; it does not establish an order inside a time block.

Different actions on the same item remain separate event rows.

In [3]:
raw = pd.read_csv(RAW_FILE, dtype=DTYPES)
assert raw.columns.tolist() == FIELDS
assert raw[["timestamp", "visitorid", "event", "itemid"]].notna().all().all()
assert set(raw["event"]) == set(EVENT_TYPES)
assert raw.loc[raw["event"].eq("transaction"), "transactionid"].notna().all()

clean = raw.drop_duplicates(FIELDS, keep="first").copy()
clean["source_row_number"] = clean.index + 2
previous = pd.read_csv(PREVIOUS_FILE, dtype=DTYPES)
pd.testing.assert_frame_equal(clean[FIELDS].reset_index(drop=True), previous[FIELDS].reset_index(drop=True))
del previous
assert len(raw) == 2_756_101
assert len(clean) == 2_755_641
assert clean["event"].eq("transaction").sum() == 22_457
assert not clean.duplicated(FIELDS).any()
print(f"Original events: {len(raw):,}; extra exact copies removed: {len(raw) - len(clean):,}")

Original events: 2,756,101; extra exact copies removed: 460


## 3. Build the three linked tables

The function below returns:
- **events_with_blocks**: every retained event, its source row, and its block ID;
- **block_items**: each distinct item in each block, listed once;
- **block_summary**: the visitor, timestamp, UTC date/time, and counts for each block.

For example, viewing and adding item 42 to a cart at the same time creates two event rows but one block–item membership.

**groupby** gathers events sharing visitor and timestamp. **agg** counts them. The text ID uses that same pair, so shuffling source rows cannot change block identities.

Sorting item IDs makes files reproducible. It is only storage/display order; the model must treat items inside a block as an unordered set.

In [4]:
def build_basket_tables(source_events):
    if source_events.empty:
        raise ValueError("No events to prepare.")
    required = FIELDS + ["source_row_number"]
    if not set(required).issubset(source_events.columns):
        raise ValueError("Missing event fields or source-row numbers.")
    if source_events[KEYS + ["itemid", "event"]].isna().any().any():
        raise ValueError("Missing required block fields.")
    if not set(source_events["event"]).issubset(EVENT_TYPES):
        raise ValueError("Unexpected event type.")

    events = source_events[required].copy()
    events["block_id"] = (
        "v" + events["visitorid"].astype("string")
        + "_t" + events["timestamp"].astype("string")
    )
    # Storage order only: never interpret itemid or source row as event timing.
    events = events.sort_values(
        KEYS + ["itemid", "event", "source_row_number"], kind="stable"
    ).reset_index(drop=True)
    members = events[["block_id", "itemid"]].drop_duplicates().reset_index(drop=True)

    counts_input = events.assign(
        view_count=events["event"].eq("view").astype("int8"),
        addtocart_count=events["event"].eq("addtocart").astype("int8"),
        transaction_count=events["event"].eq("transaction").astype("int8"),
    )
    blocks = counts_input.groupby(KEYS, as_index=False, sort=True).agg(
        block_id=("block_id", "first"),
        event_count=("itemid", "size"),
        distinct_item_count=("itemid", "nunique"),
        view_count=("view_count", "sum"),
        addtocart_count=("addtocart_count", "sum"),
        transaction_count=("transaction_count", "sum"),
        transaction_id_count=("transactionid", "count"),
        distinct_transaction_ids=("transactionid", "nunique"),
    )
    blocks["datetime_utc"] = pd.to_datetime(blocks["timestamp"], unit="ms", utc=True)
    blocks["block_type"] = "other_multi_item_block"
    same_item = blocks["event_count"].gt(1) & blocks["distinct_item_count"].eq(1)
    all_purchases = blocks["transaction_count"].eq(blocks["event_count"])
    valid_id = (blocks["transaction_id_count"].eq(blocks["event_count"])
                & blocks["distinct_transaction_ids"].eq(1))
    multi_item = blocks["distinct_item_count"].gt(1)
    blocks.loc[all_purchases & multi_item & ~valid_id, "block_type"] = "ambiguous_transaction_block"
    blocks.loc[all_purchases & multi_item & valid_id, "block_type"] = "transaction_basket"
    blocks.loc[same_item, "block_type"] = "same_item_tie"
    blocks.loc[blocks["event_count"].eq(1), "block_type"] = "single_event_block"
    blocks = blocks[[
        "block_id", "visitorid", "timestamp", "datetime_utc", "event_count",
        "distinct_item_count", "view_count", "addtocart_count", "transaction_count",
        "transaction_id_count", "distinct_transaction_ids", "block_type",
    ]]
    return events, members, blocks

## 4. Check the grouping on a tiny example

These small checks cover different items at the same time, two actions on the same item, two visitors at the same time, and an item revisited later. They also verify that missing/conflicting transaction IDs do not create a confirmed purchase basket.

The shuffled-input test compares outputs exactly. It checks that file order does not become a fake clock.

In [5]:
toy = pd.DataFrame([
    [1000, 1, "transaction", 10, 99],
    [1000, 1, "transaction", 11, 99],
    [2000, 1, "view", 10, None],
    [2000, 1, "addtocart", 10, None],
    [1000, 2, "view", 10, None],
    [3000, 1, "view", 10, None],
    [4000, 1, "view", 12, None],
    [4000, 1, "view", 13, None],
    [5000, 1, "transaction", 14, 100],
    [5000, 1, "transaction", 15, 101],
    [6000, 1, "transaction", 16, None],
    [6000, 1, "transaction", 17, None],
], columns=FIELDS).astype(DTYPES)
toy["source_row_number"] = range(2, len(toy) + 2)
toy_events, toy_members, toy_blocks = build_basket_tables(toy)
toy_lookup = toy_blocks.set_index("block_id")
assert len(toy_events) == 12 and len(toy_members) == 11 and len(toy_blocks) == 7
assert toy_lookup.loc["v1_t1000", "block_type"] == "transaction_basket"
assert toy_lookup.loc["v1_t2000", "distinct_item_count"] == 1
assert toy_lookup.loc["v1_t2000", "event_count"] == 2
assert toy_lookup.loc["v1_t5000", "block_type"] == "ambiguous_transaction_block"
assert toy_lookup.loc["v1_t6000", "block_type"] == "ambiguous_transaction_block"
assert "v2_t1000" in toy_lookup.index and "v1_t3000" in toy_lookup.index
shuffled = build_basket_tables(toy.sample(frac=1, random_state=7))
for original, reordered in zip((toy_events, toy_members, toy_blocks), shuffled):
    pd.testing.assert_frame_equal(original, reordered)
print("Small-example checks passed.")

Small-example checks passed.


## 5. Build and validate the full prepared dataset

This cell does the main work. A membership is an item belonging to a time block. Assertions stop the notebook if events disappear, block counts do not match, or a source-row link is wrong.

The minimum-five rule is deliberately deferred until training-period eligibility is defined. Retaining an event here does not promise it will be eligible for every later experiment.

In [6]:
events_with_blocks, block_items, block_summary = build_basket_tables(clean)
assert len(events_with_blocks) == 2_755_641
assert len(block_summary) == 2_753_015
assert events_with_blocks["event"].eq("transaction").sum() == 22_457
assert not block_summary["block_id"].duplicated().any()
assert not block_summary.duplicated(KEYS).any()
assert not block_items.duplicated(["block_id", "itemid"]).any()
assert events_with_blocks["block_id"].notna().all()
assert events_with_blocks["block_id"].isin(block_summary["block_id"]).all()
assert events_with_blocks.groupby("block_id")[KEYS].nunique().eq(1).all().all()

summary_index = block_summary.set_index("block_id")
pd.testing.assert_series_equal(
    events_with_blocks.groupby("block_id").size().sort_index(),
    summary_index["event_count"].sort_index(), check_names=False,
)
pd.testing.assert_series_equal(
    block_items.groupby("block_id").size().sort_index(),
    summary_index["distinct_item_count"].sort_index(), check_names=False,
)
assert block_summary["event_count"].sum() == len(events_with_blocks)
assert block_summary["distinct_item_count"].sum() == len(block_items)
assert block_summary[["view_count", "addtocart_count", "transaction_count"]].sum(axis=1).eq(
    block_summary["event_count"]
).all()

tied_blocks = block_summary.loc[block_summary["event_count"].gt(1)]
assert len(tied_blocks) == 2_143 and tied_blocks["event_count"].sum() == 4_769
assert not events_with_blocks["source_row_number"].duplicated().any()
positions = events_with_blocks["source_row_number"].to_numpy() - 2
assert (positions >= 0).all() and (positions < len(raw)).all()
pd.testing.assert_frame_equal(
    events_with_blocks[FIELDS].reset_index(drop=True),
    raw.iloc[positions][FIELDS].reset_index(drop=True),
)
print("Full dataset checks passed. Outputs are ready to export.")
display(block_summary.head())

Full dataset checks passed. Outputs are ready to export.


,block_id,visitorid,timestamp,datetime_utc,event_count,distinct_item_count,view_count,addtocart_count,transaction_count,transaction_id_count,distinct_transaction_ids,block_type
0,v0_t1442004589439,0,1442004589439,2015-09-11 20:49:49.439000+00:00,1,1,1,0,0,0,0,single_event_block
1,v0_t1442004759591,0,1442004759591,2015-09-11 20:52:39.591000+00:00,1,1,1,0,0,0,0,single_event_block
2,v0_t1442004917175,0,1442004917175,2015-09-11 20:55:17.175000+00:00,1,1,1,0,0,0,0,single_event_block
3,v1_t1439487966444,1,1439487966444,2015-08-13 17:46:06.444000+00:00,1,1,1,0,0,0,0,single_event_block
4,v2_t1438969904567,2,1438969904567,2015-08-07 17:51:44.567000+00:00,1,1,1,0,0,0,0,single_event_block


## 6. Prepare count reports and review examples

The count report describes raw events and prepared events. The block-size table counts how many blocks have each combination of event count and distinct-item count.

A **merge** joins block details onto audit event rows using block_id. Its many-to-one check ensures an event cannot accidentally be multiplied by duplicate summary rows. Example files contain complete blocks, so you can count them in Excel.

In [7]:
def population_counts(name, frame):
    return {
        "stage": name, "events": len(frame), "visitors": frame["visitorid"].nunique(),
        "items": frame["itemid"].nunique(), "blocks": len(frame[KEYS].drop_duplicates()),
        "transactions": int(frame["event"].eq("transaction").sum()),
    }

count_report = pd.DataFrame([
    population_counts("raw", raw), population_counts("prepared", events_with_blocks),
])
block_size_distribution = (
    block_summary.groupby(["event_count", "distinct_item_count"], as_index=False)
    .size().rename(columns={"size": "blocks"})
)
audit_fields = ["block_id", "source_row_number", *FIELDS,
                "event_count", "distinct_item_count", "block_type"]
audit_events = events_with_blocks.loc[events_with_blocks["block_id"].isin(tied_blocks["block_id"])]
all_tied_events = audit_events.merge(
    tied_blocks[["block_id", "event_count", "distinct_item_count", "block_type"]],
    on="block_id", validate="many_to_one",
)[audit_fields]
assert len(all_tied_events) == 4_769

# Select up to three whole blocks of each type. These are examples, not a random sample.
example_blocks = block_summary.groupby("block_type", sort=True).head(3)
examples = events_with_blocks.loc[events_with_blocks["block_id"].isin(example_blocks["block_id"])].merge(
    example_blocks[["block_id", "event_count", "distinct_item_count", "block_type"]],
    on="block_id", validate="many_to_one",
)[audit_fields]
assert examples.groupby("block_id").size().eq(
    example_blocks.set_index("block_id")["event_count"]
).all()
display(count_report)
display(block_summary["block_type"].value_counts().rename("blocks"))
display(examples.head(12))

,stage,events,visitors,items,blocks,transactions
0,raw,2756101,1407580,235061,2753015,22457
1,prepared,2755641,1407580,235061,2753015,22457


block_type
single_event_block        2750872
transaction_basket           1633
other_multi_item_block        499
same_item_tie                  11
Name: blocks, dtype: int64

,block_id,source_row_number,timestamp,visitorid,event,itemid,transactionid,event_count,distinct_item_count,block_type
0,v0_t1442004589439,1361689,1442004589439,0,view,285930,<NA>,1,1,single_event_block
1,v0_t1442004759591,1367214,1442004759591,0,view,357564,<NA>,1,1,single_event_block
2,v0_t1442004917175,1367344,1442004917175,0,view,67045,<NA>,1,1,single_event_block
3,v2799_t1440099361419,964037,1440099361419,2799,view,332761,<NA>,2,2,other_multi_item_block
4,v2799_t1440099361419,968313,1440099361419,2799,view,389853,<NA>,2,2,other_multi_item_block
5,v3465_t1434418664698,268761,1434418664698,3465,transaction,8523,11600,3,3,transaction_basket
6,v3465_t1434418664698,280708,1434418664698,3465,transaction,114485,11600,3,3,transaction_basket
7,v3465_t1434418664698,272604,1434418664698,3465,transaction,434048,11600,3,3,transaction_basket
8,v5703_t1438030995619,2649833,1438030995619,5703,transaction,186214,3297,2,2,transaction_basket
9,v5703_t1438030995619,2664130,1438030995619,5703,transaction,295495,3297,2,2,transaction_basket


## 7. Export into this run's new folders

The three main CSVs have more rows than one Excel worksheet can hold. The manifest labels that limitation and lists each output's size and row count.

Use the small audit CSVs for ordinary Excel inspection. For full files, use pandas or Excel Power Query with a filtered query; do not open the complete CSV directly and assume Excel loaded every row.

The manifest is marked complete only after all files are written and the two input data checksums still match.

In [8]:
exports = [
    (DATA_DIR / "events_with_blocks.csv", events_with_blocks),
    (DATA_DIR / "block_items.csv", block_items),
    (DATA_DIR / "block_summary.csv", block_summary),
    (REPORT_DIR / "before_after_counts.csv", count_report),
    (REPORT_DIR / "block_size_distribution.csv", block_size_distribution),
    (REPORT_DIR / "all_tied_block_events.csv", all_tied_events),
    (REPORT_DIR / "block_examples.csv", examples),
]
for path, frame in exports:
    # Exclusive creation also prevents accidental overwrites when this cell is rerun.
    frame.to_csv(path, index=False, mode="x")
    manifest["outputs"].append({
        "path": str(path), "rows": len(frame), "bytes": path.stat().st_size,
        "fits_excel_worksheet_with_header": bool(len(frame) <= 1_048_575),
    })
for path in [RAW_FILE, PREVIOUS_FILE]:
    assert file_sha256(path) == input_checksums[str(path.relative_to(PROJECT_ROOT))], (
        f"Input changed during this run: {path}"
    )
manifest["elapsed_seconds"] = time.perf_counter() - STARTED
manifest["completed_utc"] = datetime.now(timezone.utc).isoformat()
manifest["status"] = "prepared data; not yet training-ready"
manifest["validation"] = {
    "expected_counts": "passed", "source_row_mapping": "passed",
    "event_and_membership_reconciliation": "passed",
    "grouping_and_shuffle_examples": "passed", "input_data_unchanged": "passed",
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Preparation completed. This is not a train/test split.")
display(pd.DataFrame(manifest["outputs"]))
print("Manifest:", MANIFEST_FILE)

Preparation completed. This is not a train/test split.


,path,rows,bytes,fits_excel_worksheet_with_header
0,E:\ALL MCP Servers\Recommendation\data\interim...,2755641,181863536,False
1,E:\ALL MCP Servers\Recommendation\data\interim...,2755630,85352989,False
2,E:\ALL MCP Servers\Recommendation\data\interim...,2753015,306712810,False
3,E:\ALL MCP Servers\Recommendation\reports\bask...,2,138,True
4,E:\ALL MCP Servers\Recommendation\reports\bask...,7,104,True
5,E:\ALL MCP Servers\Recommendation\reports\bask...,4769,469782,True
6,E:\ALL MCP Servers\Recommendation\reports\bask...,22,2063,True


Manifest: E:\ALL MCP Servers\Recommendation\reports\basket_prepared_20261014_134756_146630\manifest.json


## Manual Excel checks

Open **block_examples.csv** or **all_tied_block_events.csv**, select the data, and press Ctrl+T.

- Filter column A (block_id). Columns D and C (visitorid and timestamp) must be identical inside the block.
- Event rows stay separate. Item ID is column F. In a same-item tie, different actions can share that item.
- Outside the table, use `=COUNTIF(A:A,A2)` for a block's event-row count. Compare it with column H (event_count).
- To count distinct items manually, copy the filtered block's item IDs to a separate scratch sheet and use Data → Remove Duplicates on that scratch copy only. Compare its count with column I (distinct_item_count).
- For a transaction basket, all event values in column E are transaction; transaction ID in G is shared and nonblank; there are multiple item IDs.
- Column B is the raw CSV source row. The full raw CSV exceeds Excel's row limit; use Power Query or pandas to locate rows beyond that limit.
- In **before_after_counts.csv**, transactions should remain 22,457. Only the 460 extra exact-copy rows disappear from the event total.

## Where we stop

We now have event records, item memberships, and time blocks. No internal order has been assigned to basket items.

The next notebook must set chronological train/validation/test boundaries, define training-only eligibility and the item catalog, and report losses from those decisions. The older strict and whole-timeline 5-core files remain historical exploratory outputs.

Both models will need an explicitly documented way to represent unordered history blocks and evaluate multiple valid target items. This preparation alone does not make the data compatible with unmodified SASRec or TiSASRec.

# Train,Test,Val Splitting

In [9]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"E:\ALL MCP Servers\Recommendation")

events = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "interim"
    / "basket_prepared_20261013_180506_369663"
    / "events_with_blocks.csv"
)

events.head()

,timestamp,visitorid,event,itemid,transactionid,source_row_number,block_id
0,1442004589439,0,view,285930,NaN,1361689,v0_t1442004589439
1,1442004759591,0,view,357564,NaN,1367214,v0_t1442004759591
2,1442004917175,0,view,67045,NaN,1367344,v0_t1442004917175
3,1439487966444,1,view,72028,NaN,830387,v1_t1439487966444
4,1438969904567,2,view,325215,NaN,742618,v2_t1438969904567


In [10]:
blocks = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "interim"
    / "basket_prepared_20261013_180506_369663"
    / "block_summary.csv"
)

In [11]:
block_items = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "interim"
    / "basket_prepared_20261013_180506_369663"
    / "block_items.csv"
)

print("Events:", len(events))
print("Blocks:", len(blocks))
print("Block-item pairs:", len(block_items))

print("\nEvent columns:")
print(events.columns.tolist())

print("\nBlock columns:")
print(blocks.columns.tolist())

assert len(events) == 2_755_641
assert len(blocks) == 2_753_015
assert len(block_items) == 2_755_630

assert events["block_id"].notna().all()
assert blocks["block_id"].is_unique
assert not block_items.duplicated(["block_id", "itemid"]).any()

print("\n✅ Files loaded and basic checks passed.")

Events: 2755641
Blocks: 2753015
Block-item pairs: 2755630

Event columns:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', 'source_row_number', 'block_id']

Block columns:
['block_id', 'visitorid', 'timestamp', 'datetime_utc', 'event_count', 'distinct_item_count', 'view_count', 'addtocart_count', 'transaction_count', 'transaction_id_count', 'distinct_transaction_ids', 'block_type']

✅ Files loaded and basic checks passed.


In [12]:
import math

# Sort only by time for the split.
# visitorid and block_id make the order stable when timestamps match.
blocks_sorted = (
    blocks
    .sort_values(["timestamp", "visitorid", "block_id"], kind="stable")
    .reset_index(drop=True)
)

assert blocks_sorted["timestamp"].is_monotonic_increasing

total_blocks = len(blocks_sorted)

# Candidate locations for 70% and 85% of all blocks
train_candidate_row = math.floor(total_blocks * 0.70) - 1
validation_candidate_row = math.floor(total_blocks * 0.85) - 1

# Use timestamps as cutoffs.
# This keeps every block with the same timestamp in the same split.
train_cutoff_timestamp = blocks_sorted.loc[train_candidate_row, "timestamp"]
validation_cutoff_timestamp = blocks_sorted.loc[
    validation_candidate_row, "timestamp"
]

assert train_cutoff_timestamp < validation_cutoff_timestamp

# Show the cutoff dates in a readable form
print(
    "Train cutoff:",
    pd.to_datetime(train_cutoff_timestamp, unit="ms", utc=True)
)
print(
    "Validation cutoff:",
    pd.to_datetime(validation_cutoff_timestamp, unit="ms", utc=True)
)
print(
    "Test starts after:",
    pd.to_datetime(validation_cutoff_timestamp, unit="ms", utc=True)
)

# Preview the real number of blocks each period will receive
train_blocks_preview = blocks_sorted[
    blocks_sorted["timestamp"] <= train_cutoff_timestamp
]
validation_blocks_preview = blocks_sorted[
    (blocks_sorted["timestamp"] > train_cutoff_timestamp)
    & (blocks_sorted["timestamp"] <= validation_cutoff_timestamp)
]
test_blocks_preview = blocks_sorted[
    blocks_sorted["timestamp"] > validation_cutoff_timestamp
]

split_preview = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "blocks": [
            len(train_blocks_preview),
            len(validation_blocks_preview),
            len(test_blocks_preview),
        ],
    }
)

split_preview["percent_of_all_blocks"] = (
    split_preview["blocks"] / total_blocks * 100
).round(3)

split_preview
validation_blocks_preview.head()

Train cutoff: 2015-08-02 23:34:51.127000+00:00
Validation cutoff: 2015-08-25 22:33:41.504000+00:00
Test starts after: 2015-08-25 22:33:41.504000+00:00


,block_id,visitorid,timestamp,datetime_utc,event_count,distinct_item_count,view_count,addtocart_count,transaction_count,transaction_id_count,distinct_transaction_ids,block_type
1927110,v1124006_t1438558492704,1124006,1438558492704,2015-08-02 23:34:52.704000+00:00,1,1,1,0,0,0,0,single_event_block
1927111,v184965_t1438558493876,184965,1438558493876,2015-08-02 23:34:53.876000+00:00,1,1,1,0,0,0,0,single_event_block
1927112,v569335_t1438558494472,569335,1438558494472,2015-08-02 23:34:54.472000+00:00,1,1,1,0,0,0,0,single_event_block
1927113,v1226999_t1438558501022,1226999,1438558501022,2015-08-02 23:35:01.022000+00:00,1,1,1,0,0,0,0,single_event_block
1927114,v1315503_t1438558507020,1315503,1438558507020,2015-08-02 23:35:07.020000+00:00,1,1,1,0,0,0,0,single_event_block


In [13]:
blocks_with_split = blocks.copy()

blocks_with_split["split"] = "test"

blocks_with_split.loc[
    blocks_with_split["timestamp"] <= train_cutoff_timestamp,
    "split"
] = "train"

blocks_with_split.loc[
    (
        blocks_with_split["timestamp"] > train_cutoff_timestamp
    )
    & (
        blocks_with_split["timestamp"] <= validation_cutoff_timestamp
    ),
    "split"
] = "validation"

# Check that every block has exactly one valid split label
assert blocks_with_split["split"].isin(
    ["train", "validation", "test"]
).all()

# Check time boundaries
assert (
    blocks_with_split.loc[
        blocks_with_split["split"].eq("train"), "timestamp"
    ].max()
    <= train_cutoff_timestamp
)

assert (
    blocks_with_split.loc[
        blocks_with_split["split"].eq("validation"), "timestamp"
    ].min()
    > train_cutoff_timestamp
)

assert (
    blocks_with_split.loc[
        blocks_with_split["split"].eq("validation"), "timestamp"
    ].max()
    <= validation_cutoff_timestamp
)

assert (
    blocks_with_split.loc[
        blocks_with_split["split"].eq("test"), "timestamp"
    ].min()
    > validation_cutoff_timestamp
)

# Review block count and date range for each split
split_check = (
    blocks_with_split
    .groupby("split", as_index=False)
    .agg(
        blocks=("block_id", "size"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
)

split_check["percent_of_blocks"] = (
    split_check["blocks"] / len(blocks_with_split) * 100
).round(3)

split_check["first_date_utc"] = pd.to_datetime(
    split_check["first_timestamp"],
    unit="ms",
    utc=True,
)

split_check["last_date_utc"] = pd.to_datetime(
    split_check["last_timestamp"],
    unit="ms",
    utc=True,
)

split_check[
    [
        "split",
        "blocks",
        "percent_of_blocks",
        "first_date_utc",
        "last_date_utc",
    ]
]

,split,blocks,percent_of_blocks,first_date_utc,last_date_utc
0,test,412953,15.0,2015-08-25 22:33:42.010000+00:00,2015-09-18 02:59:47.788000+00:00
1,train,1927110,70.0,2015-05-03 03:00:04.384000+00:00,2015-08-02 23:34:51.127000+00:00
2,validation,412952,15.0,2015-08-02 23:34:52.704000+00:00,2015-08-25 22:33:41.504000+00:00


In [14]:
# One lookup: block_id → train / validation / test
split_lookup = blocks_with_split.set_index("block_id")["split"]

# Add split label to every event row
events_with_split = events.copy()
events_with_split["split"] = events_with_split["block_id"].map(split_lookup)

# Add split label to every distinct block-item pair
block_items_with_split = block_items.copy()
block_items_with_split["split"] = block_items_with_split["block_id"].map(
    split_lookup
)

# Every row must receive a label
assert events_with_split["split"].notna().all()
assert block_items_with_split["split"].notna().all()

# Each file must use only the three expected labels
expected_splits = {"train", "validation", "test"}

assert set(events_with_split["split"]) == expected_splits
assert set(block_items_with_split["split"]) == expected_splits

# A block must always keep one split label everywhere
assert (
    events_with_split.groupby("block_id")["split"].nunique().eq(1).all()
)

assert (
    block_items_with_split.groupby("block_id")["split"].nunique().eq(1).all()
)

print("✅ Split labels successfully attached to events and block-item pairs.")

✅ Split labels successfully attached to events and block-item pairs.


In [15]:
events_with_split.groupby("split").agg(
    events=("itemid", "size"),
    visitors=("visitorid", "nunique"),
    items=("itemid", "nunique"),
    purchases=("event", lambda values: values.eq("transaction").sum()),
)

,events,visitors,items,purchases
split,,,,
test,413289,233698,97570,3351
train,1929038,978949,200978,15637
validation,413314,234695,97619,3469


In [16]:
split_order = ["train", "validation", "test"]

# A timestamp must belong to only one split.
# This proves we did not cut a simultaneous time block across periods.
assert (
    blocks_with_split.groupby("timestamp")["split"]
    .nunique()
    .eq(1)
    .all()
)

# Counts must reconcile across the three splits.
assert len(blocks_with_split) == sum(
    len(blocks_with_split[blocks_with_split["split"].eq(name)])
    for name in split_order
)

assert len(events_with_split) == sum(
    len(events_with_split[events_with_split["split"].eq(name)])
    for name in split_order
)

assert len(block_items_with_split) == sum(
    len(block_items_with_split[block_items_with_split["split"].eq(name)])
    for name in split_order
)

# Create one easy-to-read audit table.
split_audit = (
    events_with_split
    .groupby("split", observed=True)
    .agg(
        events=("itemid", "size"),
        visitors=("visitorid", "nunique"),
        items=("itemid", "nunique"),
        purchases=("event", lambda values: values.eq("transaction").sum()),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
    .reindex(split_order)
)

split_audit.insert(
    1,
    "blocks",
    blocks_with_split.groupby("split")["block_id"].size().reindex(split_order),
)

split_audit["first_date_utc"] = pd.to_datetime(
    split_audit["first_timestamp"],
    unit="ms",
    utc=True,
)

split_audit["last_date_utc"] = pd.to_datetime(
    split_audit["last_timestamp"],
    unit="ms",
    utc=True,
)

split_audit = split_audit[
    [
        "blocks",
        "events",
        "visitors",
        "items",
        "purchases",
        "first_date_utc",
        "last_date_utc",
    ]
]

print("✅ Every timestamp belongs to one split only.")
split_audit

✅ Every timestamp belongs to one split only.


,blocks,events,visitors,items,purchases,first_date_utc,last_date_utc
split,,,,,,,
train,1927110,1929038,978949,200978,15637,2015-05-03 03:00:04.384000+00:00,2015-08-02 23:34:51.127000+00:00
validation,412952,413314,234695,97619,3469,2015-08-02 23:34:52.704000+00:00,2015-08-25 22:33:41.504000+00:00
test,412953,413289,233698,97570,3351,2015-08-25 22:33:42.010000+00:00,2015-09-18 02:59:47.788000+00:00


In [17]:
SPLIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
)

# Stops accidental overwrite if this folder already exists.
assert not SPLIT_DIR.exists(), (
    f"{SPLIT_DIR} already exists. "
    "Do not overwrite an earlier split run."
)

SPLIT_DIR.mkdir(parents=True)

# Save only the basket-item rows required for basket-aware training later.
for split_name in split_order:
    split_file = SPLIT_DIR / f"{split_name}_block_items.csv"

    block_items_with_split.loc[
        block_items_with_split["split"].eq(split_name)
    ].to_csv(
        split_file,
        index=False,
    )

# Save small evidence files for your project record.
split_audit.reset_index().to_csv(
    SPLIT_DIR / "split_audit.csv",
    index=False,
)

cutoff_audit = pd.DataFrame(
    {
        "cutoff_name": [
            "train_ends_at",
            "validation_ends_at",
        ],
        "timestamp": [
            train_cutoff_timestamp,
            validation_cutoff_timestamp,
        ],
    }
)

cutoff_audit["datetime_utc"] = pd.to_datetime(
    cutoff_audit["timestamp"],
    unit="ms",
    utc=True,
)

cutoff_audit.to_csv(
    SPLIT_DIR / "split_cutoffs.csv",
    index=False,
)

# Final check: saved rows must equal the rows in RAM.
for split_name in split_order:
    expected_rows = block_items_with_split["split"].eq(split_name).sum()
    saved_rows = len(
        pd.read_csv(SPLIT_DIR / f"{split_name}_block_items.csv")
    )
    assert saved_rows == expected_rows

print("✅ Saved files:")
for file_path in sorted(SPLIT_DIR.glob("*.csv")):
    print("-", file_path.name)

AssertionError: E:\ALL MCP Servers\Recommendation\data\interim\chronological_split_70_15_15 already exists. Do not overwrite an earlier split run.

In [ ]:
# Keep only the block information needed to build visitor histories.
block_information = blocks_with_split[
    [
        "block_id",
        "visitorid",
        "timestamp",
        "datetime_utc",
        "split",
    ]
].copy()

for split_name in split_order:
    source_file = SPLIT_DIR / f"{split_name}_block_items.csv"
    output_file = SPLIT_DIR / f"{split_name}_basket_items.csv"

    # Avoid overwriting a completed output by mistake.
    assert not output_file.exists(), f"Already exists: {output_file}"

    split_items = pd.read_csv(source_file)

    enriched_split_items = split_items.merge(
        block_information,
        on=["block_id", "split"],
        how="left",
        validate="many_to_one",
    )

    enriched_split_items = (
        enriched_split_items[
            [
                "visitorid",
                "timestamp",
                "datetime_utc",
                "block_id",
                "itemid",
                "split",
            ]
        ]
        .sort_values(
            ["visitorid", "timestamp", "itemid"],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    # Verify before saving.
    assert len(enriched_split_items) == len(split_items)
    assert enriched_split_items[
        ["visitorid", "timestamp", "datetime_utc", "split"]
    ].notna().all().all()

    assert enriched_split_items["split"].eq(split_name).all()

    assert (
        enriched_split_items.groupby("block_id")[
            ["visitorid", "timestamp", "split"]
        ]
        .nunique()
        .eq(1)
        .all()
        .all()
    )

    enriched_split_items.to_csv(output_file, index=False)

    print(f"Saved: {output_file.name} | rows: {len(enriched_split_items):,}")

In [ ]:
for split_name in split_order:
    output_file = SPLIT_DIR / f"{split_name}_basket_items.csv"
    check = pd.read_csv(output_file)

    assert check.columns.tolist() == [
        "visitorid",
        "timestamp",
        "datetime_utc",
        "block_id",
        "itemid",
        "split",
    ]

    assert check["split"].eq(split_name).all()
    assert check[["visitorid", "timestamp", "block_id", "itemid"]].notna().all().all()

    assert (
        check.groupby("block_id")[["visitorid", "timestamp", "split"]]
        .nunique()
        .eq(1)
        .all()
        .all()
    )

    print(f"\n{split_name.upper()} preview:")
    display(check.head())

print("\n✅ All three basket-item files passed disk checks.")

## Splitting Done

# We’re starting the training-data reframe.

In [19]:
BASKET_FILES_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "Final basket dataset"
)

train_baskets = pd.read_csv(
    BASKET_FILES_DIR / "train_basket_items.csv"
)

# Each item should appear at most once inside one block.
assert not train_baskets.duplicated(["block_id", "itemid"]).any()

assert train_baskets["split"].eq("train").all()

# Count training time blocks per visitor and per item.
visitor_train_counts = (
    train_baskets
    .groupby("visitorid")["block_id"]
    .nunique()
    .rename("training_blocks")
)

item_train_counts = (
    train_baskets
    .groupby("itemid")["block_id"]
    .nunique()
    .rename("training_blocks")
)

print("Training basket-item rows:", len(train_baskets))
print("Training visitors:", visitor_train_counts.size)
print("Training items:", item_train_counts.size)

print("\nVisitor training-block count summary:")
display(visitor_train_counts.describe())

print("\nItem training-block count summary:")
display(item_train_counts.describe())

Training basket-item rows: 1929031
Training visitors: 978949
Training items: 200978

Visitor training-block count summary:


count    978949.000000
mean          1.968550
std          11.240468
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        5183.000000
Name: training_blocks, dtype: float64


Item training-block count summary:


count    200978.00000
mean          9.59822
std          28.20921
min           1.00000
25%           1.00000
50%           3.00000
75%           7.00000
max        1887.00000
Name: training_blocks, dtype: float64

In [ ]:
thresholds = [1, 2, 3, 5, 10]

eligibility_preview = pd.DataFrame(
    {
        "minimum_training_blocks": thresholds,
        "eligible_visitors": [
            visitor_train_counts.ge(threshold).sum()
            for threshold in thresholds
        ],
        "eligible_items": [
            item_train_counts.ge(threshold).sum()
            for threshold in thresholds
        ],
    }
)

eligibility_preview["visitor_coverage_percent"] = (
    eligibility_preview["eligible_visitors"]
    / visitor_train_counts.size
    * 100
).round(2)

eligibility_preview["item_coverage_percent"] = (
    eligibility_preview["eligible_items"]
    / item_train_counts.size
    * 100
).round(2)

eligibility_preview

In [ ]:
# One row per visitor–basket block
train_blocks_per_visitor = (
    train_baskets[
        ["visitorid", "timestamp", "block_id"]
    ]
    .drop_duplicates()
    .groupby("visitorid")["block_id"]
    .nunique()
    .rename("training_baskets")
)

# A visitor needs at least two baskets:
# first basket = history, second basket = target.
visitors_with_examples = train_blocks_per_visitor.ge(2)

number_of_eligible_visitors = visitors_with_examples.sum()

# A visitor with N baskets can form N - 1 next-basket examples.
number_of_training_examples = (
    train_blocks_per_visitor.loc[visitors_with_examples] - 1
).sum()

example_readiness = pd.DataFrame(
    {
        "metric": [
            "All training visitors",
            "Visitors with at least 2 training baskets",
            "Visitors with only 1 training basket",
            "Possible next-basket training examples",
        ],
        "value": [
            len(train_blocks_per_visitor),
            number_of_eligible_visitors,
            (~visitors_with_examples).sum(),
            number_of_training_examples,
        ],
    }
)

example_readiness["percent_of_all_visitors"] = (
    example_readiness["value"]
    / len(train_blocks_per_visitor)
    * 100
).round(2)

display(example_readiness)

# See the number of visitors at each small basket count.
training_basket_distribution = (
    train_blocks_per_visitor
    .value_counts()
    .sort_index()
    .rename_axis("training_baskets")
    .reset_index(name="visitors")
)

display(training_basket_distribution.head(10))

In [ ]:
# Load the enriched training basket file.
train_baskets = pd.read_csv(
    BASKET_FILES_DIR / "train_basket_items.csv"
)

assert train_baskets["split"].eq("train").all()
assert not train_baskets.duplicated(["block_id", "itemid"]).any()

# Rule 1: a visitor needs two real training baskets
# to create at least one history → target example.
visitor_training_baskets = (
    train_baskets
    .groupby("visitorid")["block_id"]
    .nunique()
)

eligible_train_visitors = set(
    visitor_training_baskets[
        visitor_training_baskets.ge(2)
    ].index
)

# Rule 2 for now: every item seen at least once in training
# receives a model index. Index 0 stays reserved for padding.
train_item_ids = sorted(train_baskets["itemid"].unique())

item_id_mapping = pd.DataFrame(
    {
        "itemid": train_item_ids,
        "item_index": range(1, len(train_item_ids) + 1),
    }
)

# Attach the model index to each training item.
train_encoded = train_baskets.merge(
    item_id_mapping,
    on="itemid",
    how="left",
    validate="many_to_one",
)

# Keep only visitors who can form a sequential example.
train_model_source = train_encoded.loc[
    train_encoded["visitorid"].isin(eligible_train_visitors)
].copy()

# Checks
assert train_encoded["item_index"].notna().all()
assert train_encoded["item_index"].ge(1).all()
assert train_model_source["visitorid"].nunique() == len(
    eligible_train_visitors
)

print("Training items in model catalogue:", len(item_id_mapping))
print("Visitors eligible for sequential training:", len(eligible_train_visitors))
print("Training basket-item rows after visitor rule:", len(train_model_source))

train_model_source.info()

In [ ]:
# Load validation data.
validation_baskets = pd.read_csv(
    BASKET_FILES_DIR / "validation_basket_items.csv"
)

assert validation_baskets["split"].eq("validation").all()
assert not validation_baskets.duplicated(["block_id", "itemid"]).any()

# Apply the item-ID mapping learned from training.
# New validation items will have no item_index yet.
validation_encoded = validation_baskets.merge(
    item_id_mapping,
    on="itemid",
    how="left",
    validate="many_to_one",
)

validation_encoded["item_known_to_train"] = (
    validation_encoded["item_index"].notna()
)

# Nullable integer keeps unknown item indices as <NA>.
validation_encoded["item_index"] = (
    validation_encoded["item_index"].astype("Int64")
)

# Checks
assert len(validation_encoded) == len(validation_baskets)
assert validation_encoded["split"].eq("validation").all()
assert validation_encoded["item_known_to_train"].eq(
    validation_encoded["item_index"].notna()
).all()

validation_mapping_check = pd.DataFrame(
    {
        "metric": [
            "Validation basket-item rows",
            "Rows with items known from training",
            "Rows with new items unseen in training",
            "Known-item coverage percent",
        ],
        "value": [
            len(validation_encoded),
            validation_encoded["item_known_to_train"].sum(),
            (~validation_encoded["item_known_to_train"]).sum(),
            round(
                validation_encoded["item_known_to_train"].mean() * 100,
                2,
            ),
        ],
    }
)

display(validation_mapping_check)

validation_encoded.head()

In [20]:
# Load test data.
test_baskets = pd.read_csv(
    BASKET_FILES_DIR / "test_basket_items.csv"
)

assert test_baskets["split"].eq("test").all()
assert not test_baskets.duplicated(["block_id", "itemid"]).any()

# Apply the same item-ID mapping learned from training.
test_encoded = test_baskets.merge(
    item_id_mapping,
    on="itemid",
    how="left",
    validate="many_to_one",
)

test_encoded["item_known_to_train"] = (
    test_encoded["item_index"].notna()
)

test_encoded["item_index"] = (
    test_encoded["item_index"].astype("Int64")
)

# Checks
assert len(test_encoded) == len(test_baskets)
assert test_encoded["split"].eq("test").all()
assert test_encoded["item_known_to_train"].eq(
    test_encoded["item_index"].notna()
).all()

test_mapping_check = pd.DataFrame(
    {
        "metric": [
            "Test basket-item rows",
            "Rows with items known from training",
            "Rows with new items unseen in training",
            "Known-item coverage percent",
        ],
        "value": [
            len(test_encoded),
            test_encoded["item_known_to_train"].sum(),
            (~test_encoded["item_known_to_train"]).sum(),
            round(
                test_encoded["item_known_to_train"].mean() * 100,
                2,
            ),
        ],
    }
)

display(test_mapping_check)

test_encoded.info()

NameError: name 'item_id_mapping' is not defined

In [32]:
coverage_comparison = pd.DataFrame(
    {
        "split": ["validation", "test"],
        "total_rows": [
            len(validation_encoded),
            len(test_encoded),
        ],
        "known_item_rows": [
            validation_encoded["item_known_to_train"].sum(),
            test_encoded["item_known_to_train"].sum(),
        ],
        "unseen_item_rows": [
            (~validation_encoded["item_known_to_train"]).sum(),
            (~test_encoded["item_known_to_train"]).sum(),
        ],
    }
)

coverage_comparison["known_item_percent"] = (
    coverage_comparison["known_item_rows"]
    / coverage_comparison["total_rows"]
    * 100
).round(2)

coverage_comparison

,split,total_rows,known_item_rows,unseen_item_rows,known_item_percent
0,validation,413311,385393,27918,93.25
1,test,413288,368550,44738,89.18


In [36]:
# Every training item is known because the item mapping was created from train.
train_encoded = train_encoded.copy()
train_encoded["item_known_to_train"] = True

assert train_encoded["item_index"].notna().all()
assert train_encoded["item_known_to_train"].all()

# Combine all three periods while preserving their split labels.
all_encoded = pd.concat(
    [
        train_encoded,
        validation_encoded,
        test_encoded,
    ],
    ignore_index=True,
)

assert not all_encoded.duplicated(["block_id", "itemid"]).any()
assert all_encoded["item_known_to_train"].notna().all()

# Keep items that exist in the training catalogue.
known_model_rows = all_encoded.loc[
    all_encoded["item_known_to_train"]
].copy()

assert known_model_rows["item_index"].notna().all()

# One row = one visitor basket.
known_basket_table = (
    known_model_rows
    .groupby(
        [
            "visitorid",
            "timestamp",
            "datetime_utc",
            "block_id",
            "split",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        item_indices=(
            "item_index",
            lambda values: tuple(sorted(values.astype(int))),
        ),
        item_ids=(
            "itemid",
            lambda values: tuple(sorted(values.astype(int))),
        ),
        known_item_count=("itemid", "size"),
    )
    .sort_values(
        ["visitorid", "timestamp"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert known_basket_table["block_id"].is_unique
assert known_basket_table["known_item_count"].ge(1).all()

print("✅ Modelable baskets:", len(known_basket_table))

display(known_basket_table.head(10))

✅ Modelable baskets: 2680431


,visitorid,timestamp,datetime_utc,block_id,split,item_indices,item_ids,known_item_count
0,0,1442004589439,2015-09-11 20:49:49.439000+00:00,v0_t1442004589439,test,"(123222,)","(285930,)",1
1,0,1442004759591,2015-09-11 20:52:39.591000+00:00,v0_t1442004759591,test,"(154167,)","(357564,)",1
2,0,1442004917175,2015-09-11 20:55:17.175000+00:00,v0_t1442004917175,test,"(28938,)","(67045,)",1
3,1,1439487966444,2015-08-13 17:46:06.444000+00:00,v1_t1439487966444,validation,"(31122,)","(72028,)",1
4,2,1438969904567,2015-08-07 17:51:44.567000+00:00,v2_t1438969904567,validation,"(140024,)","(325215,)",1
5,2,1438970013790,2015-08-07 17:53:33.790000+00:00,v2_t1438970013790,validation,"(140024,)","(325215,)",1
6,2,1438970212664,2015-08-07 17:56:52.664000+00:00,v2_t1438970212664,validation,"(112081,)","(259884,)",1
7,2,1438970468920,2015-08-07 18:01:08.920000+00:00,v2_t1438970468920,validation,"(93296,)","(216305,)",1
8,2,1438970905669,2015-08-07 18:08:25.669000+00:00,v2_t1438970905669,validation,"(147681,)","(342816,)",1
9,2,1438971444375,2015-08-07 18:17:24.375000+00:00,v2_t1438971444375,validation,"(147681,)","(342816,)",1


In [37]:
all_basket_counts = (
    all_encoded[["block_id", "split"]]
    .drop_duplicates()
    .groupby("split")["block_id"]
    .size()
    .rename("all_baskets")
)

modelable_basket_counts = (
    known_basket_table
    .groupby("split")["block_id"]
    .size()
    .rename("baskets_with_known_items")
)

basket_coverage = pd.concat(
    [all_basket_counts, modelable_basket_counts],
    axis=1,
).reindex(["train", "validation", "test"])

basket_coverage["known_basket_percent"] = (
    basket_coverage["baskets_with_known_items"]
    / basket_coverage["all_baskets"]
    * 100
).round(2)

basket_coverage

,all_baskets,baskets_with_known_items,known_basket_percent
split,,,
train,1927110,1927110,100.00
validation,412952,385051,93.24
test,412953,368270,89.18


In [38]:
import json

# This folder stores only the two files needed for the next model stage.
REFRAME_DIR = SPLIT_DIR / "training_reframe"

assert not REFRAME_DIR.exists(), (
    f"{REFRAME_DIR} already exists. "
    "Choose a new name instead of overwriting it."
)

REFRAME_DIR.mkdir()

# Convert basket item tuples into JSON text for safe CSV storage.
baskets_to_save = known_basket_table.copy()

for column in ["item_indices", "item_ids"]:
    baskets_to_save[column] = baskets_to_save[column].map(
        lambda values: json.dumps(
            [int(value) for value in values]
        )
    )

# One combined basket dataset with a split column.
baskets_to_save.to_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv",
    index=False,
)

# One mapping fitted only from training items.
item_id_mapping.to_csv(
    REFRAME_DIR / "train_item_id_mapping.csv",
    index=False,
)

# Disk checks
saved_baskets = pd.read_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv"
)

saved_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

assert len(saved_baskets) == len(known_basket_table)
assert saved_baskets["block_id"].is_unique
assert set(saved_baskets["split"]) == {
    "train",
    "validation",
    "test",
}

assert len(saved_mapping) == len(item_id_mapping)
assert saved_mapping["itemid"].is_unique
assert saved_mapping["item_index"].is_unique
assert saved_mapping["item_index"].min() == 1

print("✅ Saved:")
print("-", REFRAME_DIR / "known_baskets_all_splits.csv")
print("-", REFRAME_DIR / "train_item_id_mapping.csv")

display(saved_baskets.head())

✅ Saved:
- E:\ALL MCP Servers\Recommendation\data\interim\chronological_split_70_15_15\training_reframe\known_baskets_all_splits.csv
- E:\ALL MCP Servers\Recommendation\data\interim\chronological_split_70_15_15\training_reframe\train_item_id_mapping.csv


,visitorid,timestamp,datetime_utc,block_id,split,item_indices,item_ids,known_item_count
0,0,1442004589439,2015-09-11 20:49:49.439000+00:00,v0_t1442004589439,test,[123222],[285930],1
1,0,1442004759591,2015-09-11 20:52:39.591000+00:00,v0_t1442004759591,test,[154167],[357564],1
2,0,1442004917175,2015-09-11 20:55:17.175000+00:00,v0_t1442004917175,test,[28938],[67045],1
3,1,1439487966444,2015-08-13 17:46:06.444000+00:00,v1_t1439487966444,validation,[31122],[72028],1
4,2,1438969904567,2015-08-07 17:51:44.567000+00:00,v2_t1438969904567,validation,[140024],[325215],1


In [39]:
# Load the saved basket-level dataset.
model_baskets = pd.read_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv"
)

assert model_baskets["block_id"].is_unique
assert set(model_baskets["split"]) == {
    "train",
    "validation",
    "test",
}

# Count how many known baskets each visitor has in each split.
visitor_sequence_lengths = (
    model_baskets
    .groupby(["split", "visitorid"])["block_id"]
    .size()
    .rename("basket_count")
    .reset_index()
)

# Focus on train because this determines the model's training history size.
train_sequence_lengths = visitor_sequence_lengths.loc[
    visitor_sequence_lengths["split"].eq("train"),
    "basket_count",
]

sequence_length_summary = pd.DataFrame(
    {
        "metric": [
            "Visitors with training baskets",
            "Minimum baskets",
            "Median baskets",
            "75th percentile",
            "90th percentile",
            "95th percentile",
            "99th percentile",
            "Maximum baskets",
        ],
        "basket_count": [
            train_sequence_lengths.size,
            train_sequence_lengths.min(),
            train_sequence_lengths.median(),
            train_sequence_lengths.quantile(0.75),
            train_sequence_lengths.quantile(0.90),
            train_sequence_lengths.quantile(0.95),
            train_sequence_lengths.quantile(0.99),
            train_sequence_lengths.max(),
        ],
    }
)

display(sequence_length_summary)

# Show the first 20 sequence lengths and how many visitors have each one.
train_length_distribution = (
    train_sequence_lengths
    .value_counts()
    .sort_index()
    .rename_axis("training_baskets")
    .reset_index(name="visitors")
)

display(train_length_distribution.head(20))

,metric,basket_count
0,Visitors with training baskets,978949.0
1,Minimum baskets,1.0
2,Median baskets,1.0
3,75th percentile,2.0
4,90th percentile,3.0
5,95th percentile,5.0
6,99th percentile,13.0
7,Maximum baskets,5183.0


,training_baskets,visitors
0,1,697376
1,2,141818
2,3,55151
3,4,26853
4,5,16109
5,6,9850
6,7,6703
7,8,4861
8,9,3573
9,10,2618


In [40]:
# These are the first model settings.
MIN_HISTORY_BASKETS = 1
MAX_HISTORY_BASKETS = 20

# Load the saved basket-level source.
model_baskets = pd.read_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv"
)

assert model_baskets["block_id"].is_unique

# Put each visitor's baskets in true time order.
split_rank = {
    "train": 0,
    "validation": 1,
    "test": 2,
}

model_baskets["split_rank"] = model_baskets["split"].map(split_rank)

assert model_baskets["split_rank"].notna().all()

model_baskets = (
    model_baskets
    .sort_values(
        ["visitorid", "timestamp"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Position 0 means the visitor's first known basket.
model_baskets["sequence_position"] = (
    model_baskets
    .groupby("visitorid")
    .cumcount()
)

# Check that every visitor's timeline moves forward.
previous_timestamp = (
    model_baskets
    .groupby("visitorid")["timestamp"]
    .shift()
)

assert (
    model_baskets.loc[
        previous_timestamp.notna(),
        "timestamp",
    ]
    > previous_timestamp.loc[previous_timestamp.notna()]
).all()

# Create one lightweight X/y instruction row per prediction example.
# The actual history baskets stay in model_baskets, avoiding huge repeated copies.
example_index = (
    model_baskets.loc[
        model_baskets["sequence_position"].ge(
            MIN_HISTORY_BASKETS
        ),
        [
            "visitorid",
            "split",
            "timestamp",
            "block_id",
            "sequence_position",
        ],
    ]
    .rename(
        columns={
            "timestamp": "target_timestamp",
            "block_id": "target_block_id",
            "sequence_position": "target_position",
        }
    )
    .copy()
)

# Example history starts at most 20 baskets before the target.
example_index["history_start_position"] = (
    example_index["target_position"]
    .sub(MAX_HISTORY_BASKETS)
    .clip(lower=0)
)

example_index["history_end_position"] = (
    example_index["target_position"] - 1
)

example_index["history_length"] = (
    example_index["history_end_position"]
    - example_index["history_start_position"]
    + 1
)

# Checks
assert example_index["history_length"].between(
    MIN_HISTORY_BASKETS,
    MAX_HISTORY_BASKETS,
).all()

assert example_index["target_position"].gt(
    example_index["history_end_position"]
).all()

# How many prediction examples belong to each split?
examples_by_split = (
    example_index
    .groupby("split")
    .size()
    .reindex(["train", "validation", "test"])
    .rename("examples")
    .reset_index()
)

display(examples_by_split)
display(example_index.head(10))

,split,examples
0,train,948161
1,validation,182936
2,test,173786


,visitorid,split,target_timestamp,target_block_id,target_position,history_start_position,history_end_position,history_length
1,0,test,1442004759591,v0_t1442004759591,1,0,0,1
2,0,test,1442004917175,v0_t1442004917175,2,0,1,2
5,2,validation,1438970013790,v2_t1438970013790,1,0,0,1
6,2,validation,1438970212664,v2_t1438970212664,2,0,1,2
7,2,validation,1438970468920,v2_t1438970468920,3,0,2,3
8,2,validation,1438970905669,v2_t1438970905669,4,0,3,4
9,2,validation,1438971444375,v2_t1438971444375,5,0,4,5
10,2,validation,1438971463170,v2_t1438971463170,6,0,5,6
11,2,validation,1438971657845,v2_t1438971657845,7,0,6,7
16,6,test,1440916778318,v6_t1440916778318,1,0,0,1


In [41]:
# Each target block contains the correct multi-item basket label.
target_lookup = (
    model_baskets[
        [
            "block_id",
            "split",
            "item_indices",
            "known_item_count",
        ]
    ]
    .rename(
        columns={
            "split": "target_split_check",
            "item_indices": "target_item_indices",
            "known_item_count": "target_item_count",
        }
    )
)

examples_with_targets = example_index.merge(
    target_lookup,
    left_on="target_block_id",
    right_on="block_id",
    how="left",
    validate="one_to_one",
)

# Checks
assert len(examples_with_targets) == len(example_index)
assert examples_with_targets["target_item_indices"].notna().all()
assert examples_with_targets["target_item_count"].ge(1).all()

assert examples_with_targets["split"].eq(
    examples_with_targets["target_split_check"]
).all()

examples_with_targets = examples_with_targets.drop(
    columns=["block_id", "target_split_check"]
)

# Show the number of examples and target items in each split.
example_target_summary = (
    examples_with_targets
    .groupby("split", as_index=False)
    .agg(
        examples=("target_block_id", "size"),
        average_target_items=(
            "target_item_count",
            "mean",
        ),
        maximum_target_items=(
            "target_item_count",
            "max",
        ),
    )
)

example_target_summary["average_target_items"] = (
    example_target_summary["average_target_items"].round(3)
)

display(example_target_summary)

display(
    examples_with_targets[
        [
            "visitorid",
            "split",
            "target_timestamp",
            "history_start_position",
            "history_end_position",
            "history_length",
            "target_block_id",
            "target_item_indices",
            "target_item_count",
        ]
    ].head(10)
)

,split,examples,average_target_items,maximum_target_items
0,test,173786,1.002,6
1,train,948161,1.002,6
2,validation,182936,1.002,6


,visitorid,split,target_timestamp,history_start_position,history_end_position,history_length,target_block_id,target_item_indices,target_item_count
0,0,test,1442004759591,0,0,1,v0_t1442004759591,[154167],1
1,0,test,1442004917175,0,1,2,v0_t1442004917175,[28938],1
2,2,validation,1438970013790,0,0,1,v2_t1438970013790,[140024],1
3,2,validation,1438970212664,0,1,2,v2_t1438970212664,[112081],1
4,2,validation,1438970468920,0,2,3,v2_t1438970468920,[93296],1
5,2,validation,1438970905669,0,3,4,v2_t1438970905669,[147681],1
6,2,validation,1438971444375,0,4,5,v2_t1438971444375,[147681],1
7,2,validation,1438971463170,0,5,6,v2_t1438971463170,[93296],1
8,2,validation,1438971657845,0,6,7,v2_t1438971657845,[140024],1
9,6,test,1440916778318,0,0,1,v6_t1440916778318,[109356],1


In [42]:
EXAMPLES_FILE = REFRAME_DIR / "examples_with_targets.csv"

assert not EXAMPLES_FILE.exists(), (
    f"{EXAMPLES_FILE} already exists. "
    "Do not overwrite a completed output."
)

examples_with_targets.to_csv(
    EXAMPLES_FILE,
    index=False,
)

# Read back and verify the saved result.
saved_examples = pd.read_csv(EXAMPLES_FILE)

assert len(saved_examples) == len(examples_with_targets)

assert saved_examples["history_length"].between(
    MIN_HISTORY_BASKETS,
    MAX_HISTORY_BASKETS,
).all()

assert saved_examples["target_item_indices"].notna().all()

assert set(saved_examples["split"]) == {
    "train",
    "validation",
    "test",
}

print("✅ Saved X/y reference table:")
print(EXAMPLES_FILE)

display(saved_examples.head())

✅ Saved X/y reference table:
E:\ALL MCP Servers\Recommendation\data\interim\chronological_split_70_15_15\training_reframe\examples_with_targets.csv


,visitorid,split,target_timestamp,target_block_id,target_position,history_start_position,history_end_position,history_length,target_item_indices,target_item_count
0,0,test,1442004759591,v0_t1442004759591,1,0,0,1,[154167],1
1,0,test,1442004917175,v0_t1442004917175,2,0,1,2,[28938],1
2,2,validation,1438970013790,v2_t1438970013790,1,0,0,1,[140024],1
3,2,validation,1438970212664,v2_t1438970212664,2,0,1,2,[112081],1
4,2,validation,1438970468920,v2_t1438970468920,3,0,2,3,[93296],1
